# Post-Conversion Election Results QC & Validation

This notebook provides an interactive validation and quality-control environment to run immediately after converting raw election JSON exports to OpenElections CSVs (`county-level.csv` and `precinct-level.csv`).

### Objectives:
1. **Overall Results**: Compare candidate total votes and voting method breakouts against the election portal UI.
2. **County-Level Results**: Verify county totals and check reconciliation between county totals and precinct sums.
3. **Precinct-Level Results**: Drill down into specific counties and individual precincts to compare against local reporting displays.
4. **Integrity & Anomaly Tests**: Detect missing breakouts, math discrepancies, duplicates, or voter-privacy suppressed precincts.

In [1]:
# 1. Environment & Setup
from pathlib import Path
import duckdb
import pandas as pd

# Display settings for clean tabular viewing
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)

## 2. File Selection & Ingestion
Select the election slug to validate. The notebook will automatically look for the corresponding `county-level.csv` and `precinct-level.csv` files in the parent directory (`2025/`).

In [2]:
# Available election slugs in the 2025 directory:
# - 20250923__ga__special__state__senate__21__runoff
# - 20250826__ga__special__state__senate__21
# - 20250826__ga__recount__special__state__senate__21
# - 20250715__ga__special__runoff__psc
# - 20250617__ga__special__psc

BASE_DIR = Path("..").resolve()
ELECTION_SLUG = "20250923__ga__special__state__senate__21__runoff"

county_csv = BASE_DIR / f"{ELECTION_SLUG}__county-level.csv"
precinct_csv = BASE_DIR / f"{ELECTION_SLUG}__precinct-level.csv"
recon_csv = BASE_DIR / f"{ELECTION_SLUG}__reconciliation_report.csv"

print(f"County CSV exists:   {county_csv.exists()} ({county_csv.name})")
print(f"Precinct CSV exists: {precinct_csv.exists()} ({precinct_csv.name})")
print(f"Recon CSV exists:    {recon_csv.exists()} ({recon_csv.name})")

County CSV exists:   True (20250923__ga__special__state__senate__21__runoff__county-level.csv)
Precinct CSV exists: True (20250923__ga__special__state__senate__21__runoff__precinct-level.csv)
Recon CSV exists:    True (20250923__ga__special__state__senate__21__runoff__reconciliation_report.csv)


In [3]:
# Initialize in-memory DuckDB connection and register views
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE VIEW county_data AS SELECT * FROM read_csv_auto('{county_csv}')")
con.execute(f"CREATE OR REPLACE VIEW precinct_data AS SELECT * FROM read_csv_auto('{precinct_csv}')")

def sql(query_str):
    """Helper to run DuckDB SQL queries and return a DataFrame with integers for vote counts."""
    df = con.sql(query_str).df()
    for col in df.columns:
        # Keep percentage columns as float
        if 'pct' in col.lower() or 'percent' in col.lower():
            continue
        # Cast float vote/count/diff columns that represent whole numbers to nullable Int64
        if pd.api.types.is_float_dtype(df[col]):
            valid = df[col].dropna()
            if len(valid) > 0 and (valid % 1 == 0).all():
                df[col] = df[col].astype('Int64')
        elif col.endswith(('_votes', '_count', '_diff', 'votes', 'diff', 'count')):
            try:
                df[col] = df[col].astype('Int64')
            except Exception:
                pass
    return df

county_count = sql("SELECT COUNT(*) as count FROM county_data")['count'][0]
precinct_count = sql("SELECT COUNT(*) as count FROM precinct_data")['count'][0]
print(f"✓ Loaded {county_count:,} county-level rows and {precinct_count:,} precinct-level rows.")

✓ Loaded 4 county-level rows and 94 precinct-level rows.


## 3. Contests, Parties & Candidates Audit
Verify that offices, districts, party affiliations, and candidate names match expectations.

In [4]:
# Distinct Office, District, Party, Candidate in the dataset
sql('''
    SELECT DISTINCT office, district, party, candidate
    FROM precinct_data
    ORDER BY office, district, candidate
''')

,office,district,party,candidate
0,State Senate,21,Democrat,Debra Shigley
1,State Senate,21,Republican,Jason T. Dickerson


## 4. Overall Election Results (Compare to SOS UI Homepage)
Displays the overall candidate totals across the entire district/race. Compares certified county totals with the aggregated precinct sums and shows the countywide vote modality breakdowns (Election Day, Advance, Mail, Provisional). All vote counts are formatted as integers.

In [5]:
# Dynamic detection for county total votes
has_county_votes_col = 'votes' in [c.lower() for c in con.table('county_data').columns]
county_vote_expr = "CAST(SUM(COALESCE(votes, 0)) AS BIGINT)" if has_county_votes_col else \
    "CAST(SUM(COALESCE(election_day_votes, 0) + COALESCE(advanced_votes, 0) + COALESCE(absentee_by_mail_votes, 0) + COALESCE(provisional_votes, 0)) AS BIGINT)"

query_overall = f'''
WITH c_agg AS (
    SELECT 
        office, district, party, candidate,
        {county_vote_expr} AS county_votes,
        CAST(SUM(COALESCE(election_day_votes, 0)) AS BIGINT) AS ed_votes,
        CAST(SUM(COALESCE(advanced_votes, 0)) AS BIGINT) AS adv_votes,
        CAST(SUM(COALESCE(absentee_by_mail_votes, 0)) AS BIGINT) AS mail_votes,
        CAST(SUM(COALESCE(provisional_votes, 0)) AS BIGINT) AS prov_votes
    FROM county_data
    GROUP BY office, district, party, candidate
),
p_agg AS (
    SELECT 
        office, district, party, candidate,
        CAST(SUM(COALESCE(votes, 0)) AS BIGINT) AS precinct_votes_sum
    FROM precinct_data
    GROUP BY office, district, party, candidate
)
SELECT 
    c.office,
    c.district,
    c.party,
    c.candidate,
    c.county_votes,
    p.precinct_votes_sum,
    CAST(c.county_votes - p.precinct_votes_sum AS BIGINT) AS diff,
    ROUND(100.0 * c.county_votes / SUM(c.county_votes) OVER (PARTITION BY c.office, c.district), 2) AS pct_total,
    c.ed_votes AS election_day,
    c.adv_votes AS advanced,
    c.mail_votes AS absentee_mail,
    c.prov_votes AS provisional
FROM c_agg c
JOIN p_agg p USING (office, district, party, candidate)
ORDER BY c.county_votes DESC
'''
sql(query_overall)

,office,district,party,candidate,county_votes,precinct_votes_sum,diff,pct_total,election_day,advanced,absentee_mail,provisional
0,State Senate,21,Republican,Jason T. Dickerson,19065,19065,0,61.46,9629,9348,84,4
1,State Senate,21,Democrat,Debra Shigley,11955,11955,0,38.54,4939,6856,155,5


## 5. County-Level Summary & Reconciliation
Aggregates each county's totals to verify against the official county summary screen in the election UI. Checks if the sum of precinct votes equals the countywide certified total (`diff = 0` indicates a perfect match). All vote values are formatted as integers.

In [6]:
query_county = f'''
WITH c_sum AS (
    SELECT 
        county, office, district, party, candidate,
        {county_vote_expr} AS county_votes,
        CAST(SUM(COALESCE(election_day_votes, 0)) AS BIGINT) AS ed_votes,
        CAST(SUM(COALESCE(advanced_votes, 0)) AS BIGINT) AS adv_votes,
        CAST(SUM(COALESCE(absentee_by_mail_votes, 0)) AS BIGINT) AS mail_votes,
        CAST(SUM(COALESCE(provisional_votes, 0)) AS BIGINT) AS prov_votes
    FROM county_data
    GROUP BY county, office, district, party, candidate
),
p_sum AS (
    SELECT 
        county, office, district, party, candidate,
        CAST(SUM(COALESCE(votes, 0)) AS BIGINT) AS precinct_votes_sum,
        CAST(COUNT(DISTINCT precinct) AS BIGINT) AS precinct_count
    FROM precinct_data
    GROUP BY county, office, district, party, candidate
)
SELECT 
    c.county,
    c.candidate,
    c.party,
    c.county_votes,
    p.precinct_votes_sum,
    p.precinct_count,
    CAST(c.county_votes - p.precinct_votes_sum AS BIGINT) AS diff,
    CASE WHEN c.county_votes = p.precinct_votes_sum THEN 'MATCH' ELSE 'DISCREPANCY' END AS status,
    c.ed_votes AS county_ed,
    c.adv_votes AS county_adv,
    c.mail_votes AS county_mail,
    c.prov_votes AS county_prov
FROM c_sum c
JOIN p_sum p USING (county, office, district, party, candidate)
ORDER BY c.county, c.candidate
'''
sql(query_county)

,county,candidate,party,county_votes,precinct_votes_sum,precinct_count,diff,status,county_ed,county_adv,county_mail,county_prov
0,Cherokee,Debra Shigley,Democrat,5416,5416,18,0,MATCH,2373,2935,106,2
1,Cherokee,Jason T. Dickerson,Republican,13652,13652,18,0,MATCH,7016,6562,72,2
2,Fulton,Debra Shigley,Democrat,6539,6539,29,0,MATCH,2566,3921,49,3
3,Fulton,Jason T. Dickerson,Republican,5413,5413,29,0,MATCH,2613,2786,12,2


## 6. Interactive Drill-Down (Filter by County and/or Precinct)
Use the configuration variables below to drill down into a specific county or individual precinct. This allows you to inspect specific precincts against the county election board or SOS precinct reporting view. All vote counts and breakouts are integers (or `<NA>` if unallocated).

In [9]:
# --- FILTER CONFIGURATION ---
# Set to None to include all, or specify string values:
FILTER_COUNTY = "Cherokee"      # e.g. "Cherokee", "Fulton", or None
FILTER_PRECINCT = "Hickory Log" # e.g. "Airport", "Avery", "AP07C", or None
# FILTER_PRECINCT = None
# ----------------------------

where_clauses = []
if FILTER_COUNTY:
    where_clauses.append(f"county = '{FILTER_COUNTY}'")
if FILTER_PRECINCT:
    where_clauses.append(f"precinct = '{FILTER_PRECINCT}'")

where_sql = ("WHERE " + " AND ".join(where_clauses)) if where_clauses else ""

query_drilldown = f'''
    SELECT 
        county,
        precinct,
        candidate,
        party,
        votes,
        election_day_votes,
        advanced_votes,
        absentee_by_mail_votes,
        provisional_votes
    FROM precinct_data
    {where_sql}
    ORDER BY county, precinct, candidate
'''
df_drilldown = sql(query_drilldown)
print(f"Displaying {len(df_drilldown)} rows (County: {FILTER_COUNTY}, Precinct: {FILTER_PRECINCT}):")
df_drilldown

Displaying 2 rows (County: Cherokee, Precinct: Hickory Log):


,county,precinct,candidate,party,votes,election_day_votes,advanced_votes,absentee_by_mail_votes,provisional_votes
0,Cherokee,Hickory Log,Debra Shigley,Democrat,670,231,427,12,0
1,Cherokee,Hickory Log,Jason T. Dickerson,Republican,1164,475,679,10,0


## 7. Precinct Total Turnout Summary
Aggregates total votes cast across all candidates per precinct as integer vote counts to quickly verify precinct turnout rankings.

In [10]:
# Turnout per precinct across all candidates
query_turnout = f'''
    SELECT 
        county,
        precinct,
        CAST(SUM(votes) AS BIGINT) AS total_precinct_votes,
        CAST(COUNT(DISTINCT candidate) AS BIGINT) AS candidate_count
    FROM precinct_data
    {where_sql}
    GROUP BY county, precinct
    ORDER BY county, total_precinct_votes DESC
'''
sql(query_turnout)

,county,precinct,total_precinct_votes,candidate_count
0,Cherokee,Hickory Log,1834,2


## 8. Data Quality & Anomaly Tests
These automated checks test for common election data issues, unassigned batches, math inconsistencies, and voter privacy suppression.

### Test A: Precinct Breakout Availability Check
Checks which precincts have vote method breakout numbers (`election_day_votes`, etc.) versus those where only aggregate `votes` are reported.

In [11]:
breakout_summary = sql('''
    SELECT 
        county,
        CAST(COUNT(DISTINCT precinct) AS BIGINT) AS total_precincts,
        CAST(COUNT(DISTINCT CASE WHEN election_day_votes IS NOT NULL THEN precinct END) AS BIGINT) AS precincts_with_breakouts,
        CAST(COUNT(DISTINCT CASE WHEN election_day_votes IS NULL THEN precinct END) AS BIGINT) AS precincts_missing_breakouts,
        ROUND(100.0 * COUNT(DISTINCT CASE WHEN election_day_votes IS NOT NULL THEN precinct END) / COUNT(DISTINCT precinct), 1) AS breakout_pct
    FROM precinct_data
    GROUP BY county
    ORDER BY county
''')
display(breakout_summary)

# List precincts missing breakouts
missing_precincts = sql('''
    SELECT DISTINCT county, precinct
    FROM precinct_data
    WHERE election_day_votes IS NULL
    ORDER BY county, precinct
''')
if len(missing_precincts) > 0:
    print(f"\n⚠ {len(missing_precincts)} precinct(s) missing vote method breakouts (only total votes reported):")
    display(missing_precincts)
else:
    print("\n✓ All precincts have vote method breakouts!")

,county,total_precincts,precincts_with_breakouts,precincts_missing_breakouts,breakout_pct
0,Cherokee,18,12,6,66.7
1,Fulton,29,11,18,37.9



⚠ 24 precinct(s) missing vote method breakouts (only total votes reported):


,county,precinct
0,Cherokee,Avery
1,Cherokee,Ball Ground
2,Cherokee,Conns Creek
3,Cherokee,Hickory Flat
4,Cherokee,Mountain Road
5,Cherokee,Salacoa
6,Fulton,AP01B
7,Fulton,AP021
8,Fulton,AP022
9,Fulton,AP07A


### Test B: Breakout Math Consistency Check
For precincts where breakouts *are* reported, this verifies that:
`votes == election_day_votes + advanced_votes + absentee_by_mail_votes + provisional_votes`

In [12]:
math_check = sql('''
    SELECT 
        county, precinct, candidate, votes,
        CAST(COALESCE(election_day_votes, 0) + COALESCE(advanced_votes, 0) + COALESCE(absentee_by_mail_votes, 0) + COALESCE(provisional_votes, 0) AS BIGINT) AS breakout_sum,
        CAST(votes - (COALESCE(election_day_votes, 0) + COALESCE(advanced_votes, 0) + COALESCE(absentee_by_mail_votes, 0) + COALESCE(provisional_votes, 0)) AS BIGINT) AS diff
    FROM precinct_data
    WHERE election_day_votes IS NOT NULL
      AND votes <> (COALESCE(election_day_votes, 0) + COALESCE(advanced_votes, 0) + COALESCE(absentee_by_mail_votes, 0) + COALESCE(provisional_votes, 0))
''')
if len(math_check) == 0:
    print("✓ Pass: All reported breakout sums exactly equal the precinct total votes!")
else:
    print(f"✗ Found {len(math_check)} math discrepancies:")
    display(math_check)

✓ Pass: All reported breakout sums exactly equal the precinct total votes!


### Test C: Duplicate Record Detection
Verifies that each candidate appears at most once per precinct.

In [13]:
duplicates = sql('''
    SELECT county, precinct, office, district, candidate, CAST(COUNT(*) AS BIGINT) AS count
    FROM precinct_data
    GROUP BY county, precinct, office, district, candidate
    HAVING COUNT(*) > 1
''')
if len(duplicates) == 0:
    print("✓ Pass: No duplicate records found in precinct dataset.")
else:
    print(f"✗ Warning: Found {len(duplicates)} duplicate candidate-precinct rows:")
    display(duplicates)

✓ Pass: No duplicate records found in precinct dataset.


### Test D: Voter Privacy Suppression & Zero-Vote Audit
Detects precincts with zero votes or missing candidates, which often occurs due to voter privacy protections:  
*"Some results have been hidden in the cases where not enough ballots were cast to ensure voter privacy."*

In [14]:
suppressed_check = sql('''
    SELECT 
        county,
        precinct,
        CAST(SUM(COALESCE(votes, 0)) AS BIGINT) AS total_votes
    FROM precinct_data
    GROUP BY county, precinct
    HAVING SUM(COALESCE(votes, 0)) = 0 OR SUM(votes) IS NULL
    ORDER BY county, precinct
''')
if len(suppressed_check) == 0:
    print("✓ All precincts have non-zero recorded votes.")
else:
    print(f"ℹ Found {len(suppressed_check)} precinct(s) with 0 votes (potential voter privacy suppression):")
    display(suppressed_check)

✓ All precincts have non-zero recorded votes.


## 9. Ad-Hoc SQL Playground
You can execute any custom SQL query using the `sql(...)` helper function on `county_data` or `precinct_data`.

In [15]:
# Example ad-hoc query from qc_race_results.sql
sql('''
    SELECT 
        candidate,
        county,
        precinct,
        votes,
        election_day_votes,
        advanced_votes,
        absentee_by_mail_votes,
        provisional_votes
    FROM precinct_data
    WHERE county = 'Cherokee' AND precinct = 'Airport'
    ORDER BY candidate
''')

,candidate,county,precinct,votes,election_day_votes,advanced_votes,absentee_by_mail_votes,provisional_votes
0,Debra Shigley,Cherokee,Airport,372,151,217,4,0
1,Jason T. Dickerson,Cherokee,Airport,946,464,477,5,0
